# Calibrated Selective Re-Identification for Occlusion-Robust Object-Centric Video Representations

This notebook runs the research workflow in Google Colab. It tests whether calibrated association confidence and selective rejection reduce identity switches when an object reappears after absence.

**Scope:** DAVIS ground-truth instance masks are used as detections. This isolates association from segmentation errors; this project does not evaluate segmentation, object discovery, or general-purpose multi-object tracking.

Upload or clone this repository to `/content/occlusion-aware-object-association`, then download DAVIS 2017 TrainVal 480p from the [official DAVIS page](https://davischallenge.org/davis2017/code.html) into Google Drive. The default path is `/content/drive/MyDrive/DAVIS`. Both official train and validation split files are required.

## 1. Project introduction and research question

The proposed model estimates **P(correct association | association features)** from training-sequence candidate pairs. It accepts assignments above a fixed confidence threshold and rejects weaker assignments; validation sequences remain held out for final evaluation. The experiment reports identity tracking quality separately from probability calibration.

## 2. Check Python and CUDA environment

In [ ]:
import platform
import sys

print("Python:", platform.python_version())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        free_bytes, total_bytes = torch.cuda.mem_get_info(0)
        print(f"Available GPU memory: {free_bytes / (1024**3):.2f} GiB free / {total_bytes / (1024**3):.2f} GiB total")
    else:
        print("GPU: none; supported feature extraction falls back to CPU")
except ImportError:
    print("PyTorch is not installed in this runtime yet")

## 3. Install dependencies

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/occlusion-aware-object-association")
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(
        f"{PROJECT_DIR} does not contain requirements.txt. Upload or clone the repository there first."
    )
%pip install -q -r /content/occlusion-aware-object-association/requirements.txt
%pip install -q -e /content/occlusion-aware-object-association

## 4. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 5. Configure dataset and run mode

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

from occlusion_association.config import load_config

DAVIS_ROOT = Path("/content/drive/MyDrive/DAVIS")  # Change only this value if your Drive layout differs.
RUN_MODE = "smoke"  # Choose "smoke" for a small run or "full" for the official train/validation split.
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

config = load_config(
    PROJECT_DIR / "configs" / "default.yaml",
    overrides={
        "dataset": {"root": str(DAVIS_ROOT)},
        "experiment": {"smoke_test": RUN_MODE == "smoke"},
    },
)
print("Repository:", PROJECT_DIR)
print("DAVIS root:", config.dataset_root)
print("Run mode:", RUN_MODE)
print("Persistent feature cache:", config.features["cache_dir"])
print("Persistent experiment output:", config.experiment["output_dir"])

## 6. Validate DAVIS dataset structure

In [ ]:
from occlusion_association.dataset import DAVISDataset

dataset = DAVISDataset(
    config.dataset_root,
    split="all",
    train_split=config.dataset["train_split"],
    validation_split=config.dataset["validation_split"],
)
print("Resolved DAVIS root:", dataset.root)
print("Official train sequences:", len(dataset.train_sequences))
print("Official validation sequences:", len(dataset.validation_sequences))
print("Split files:", config.dataset["train_split"], "and", config.dataset["validation_split"])

## 7. Inspect dataset

In [ ]:
from occlusion_association.utils import select_smoke_sequence_names

if RUN_MODE == "smoke":
    train_sequence_names = select_smoke_sequence_names(
        dataset, dataset.train_sequences, int(config.experiment["smoke_test_sequences"])
    )
    validation_sequence_names = select_smoke_sequence_names(
        dataset, dataset.validation_sequences, int(config.experiment["smoke_test_sequences"])
    )
    FRAME_LIMIT = int(config.experiment["smoke_test_frames"])
else:
    train_sequence_names = list(dataset.train_sequences)
    validation_sequence_names = list(dataset.validation_sequences)
    FRAME_LIMIT = None

selected_names = train_sequence_names + validation_sequence_names
inspection = dataset.inspect(
    sequence_names=selected_names,
    frame_limit=FRAME_LIMIT,
)
for sequence in inspection.inspected_sequences:
    status = "SKIPPED" if sequence.skipped else "ok"
    print(
        f"{sequence.sequence}: {status}; valid frames={sequence.valid_frames}/"
        f"{sequence.matched_frames} matched; objects={sequence.unique_objects}; "
        f"image sizes={sequence.image_dimensions}; annotation sizes={sequence.annotation_dimensions}"
    )
    for lifecycle in sequence.object_lifecycles:
        print(
            f"  object {lifecycle['object_id']}: first={lifecycle['first_observed_frame']}, "
            f"last={lifecycle['last_observed_frame']}, valid-frame gaps={len(lifecycle['missing_gaps'])}"
        )
print("Dataset loading issues:", len(inspection.loading_errors))
for issue in inspection.loading_errors:
    print(f"[{issue.severity}:{issue.code}] {issue.sequence or ''} {issue.frame or ''}: {issue.message}")

## 8. Load frozen DINOv2 ViT-S/14

In [ ]:
from occlusion_association.features import DINOv2FeatureExtractor

extractor = DINOv2FeatureExtractor(
    backbone_name=config.features["backbone"],
    repository=config.features["torch_hub_repository"],
    input_size=int(config.features["input_size"]),
    device=config.features["device"],
    use_mixed_precision=bool(config.features["use_mixed_precision"]),
)
print("Frozen backbone:", extractor.backbone_name)
print("Device:", extractor.device)
print("Preprocessing/cache identity:", extractor.preprocessing)

## 9. Extract mask-pooled object features

In [ ]:
from occlusion_association.feature_cache import FeatureCache
from occlusion_association.features import extract_dataset_features

cache = FeatureCache(config.features["cache_dir"])
selected_names = train_sequence_names + validation_sequence_names
feature_summaries, feature_issues = extract_dataset_features(
    dataset,
    selected_names,
    extractor,
    cache,
    batch_size=int(config.features["batch_size"]),
    frame_limit=FRAME_LIMIT,
)
for summary in feature_summaries:
    print(summary)
print("Feature extraction loading issues:", len(feature_issues))
for issue in feature_issues:
    print(f"[{issue.severity}:{issue.code}] {issue.sequence or ''} {issue.frame or ''}: {issue.message}")

## 10. Cache and verify features

In [ ]:
from occlusion_association.features import load_cached_frame_embeddings

cache_check = {"frames": 0, "object_features": 0, "missing_features": 0}
cache_issues = []
for sequence_name in selected_names:
    frame_iterator = dataset.iter_frames(sequence_name, issue_sink=cache_issues)
    if FRAME_LIMIT is not None:
        from itertools import islice
        frame_iterator = islice(frame_iterator, FRAME_LIMIT)
    for frame in frame_iterator:
        embeddings, missing = load_cached_frame_embeddings(
            frame,
            cache,
            backbone_name=extractor.backbone_name,
            preprocessing=extractor.preprocessing,
        )
        cache_check["frames"] += 1
        cache_check["object_features"] += len(embeddings)
        cache_check["missing_features"] += len(missing)
print(cache_check)
if cache_check["missing_features"]:
    raise RuntimeError("Feature cache verification failed; rerun extraction before calibration.")

## 11. Generate training association examples

In [ ]:
from itertools import islice
from occlusion_association.calibration import generate_calibration_candidates

def training_streams():
    for sequence_name in train_sequence_names:
        frames = dataset.iter_frames(sequence_name)
        if FRAME_LIMIT is not None:
            frames = islice(frames, FRAME_LIMIT)
        yield sequence_name, frames

training_candidates = generate_calibration_candidates(
    training_streams(),
    cache,
    backbone_name=extractor.backbone_name,
    preprocessing=extractor.preprocessing,
    max_gap=int(config.memory["max_gap"]),
    appearance_history_size=int(config.memory["appearance_history_size"]),
)
print("Training sequence names:", sorted(training_candidates.sequence_names))
print("Candidate matrix:", training_candidates.features.shape)
print("Positive/negative targets:", int((training_candidates.labels == 1).sum()), int((training_candidates.labels == 0).sum()))

## 12. Fit the confidence calibrator

In [ ]:
from occlusion_association.calibration import fit_logistic_calibrator

calibrator = fit_logistic_calibrator(
    training_candidates,
    training_sequences=train_sequence_names,
    validation_sequences=validation_sequence_names,
    class_weight=config.calibration["class_weight"],
    max_iter=int(config.calibration["max_iter"]),
    random_state=int(config.calibration["random_state"]),
)
calibrator.metadata["validation_sequences_checked"] = validation_sequence_names
calibrator.metadata["max_memory_gap"] = int(config.memory["max_gap"])
print(calibrator.metadata)

## 13. Run the association baselines

In [ ]:
from occlusion_association.experiments import run_experiment_suite

# The runner applies identical oracle-mask detections and cached frozen features to all methods.
# It also performs the proposed method and the requested ablations in the same held-out run.
experiment_results = run_experiment_suite(
    dataset,
    cache,
    config.to_dict(),
    train_sequence_names=train_sequence_names,
    validation_sequence_names=validation_sequence_names,
    frame_limit=FRAME_LIMIT,
    training_candidates=training_candidates,
    fitted_calibrator=calibrator,
)
experiment_results["smoke_test"] = RUN_MODE == "smoke"
import pandas as pd
pd.DataFrame(
    [row for row in experiment_results["metrics"] if row["method"] != "calibrated_selective"]
)

## 14. Run the proposed selective method

In [ ]:
pd.DataFrame(
    [row for row in experiment_results["metrics"] if row["method"] == "calibrated_selective"]
)

## 15. Run ablations

In [ ]:
print("Appearance versus motion")
display(pd.DataFrame(experiment_results["appearance_ablation"]))
print("Calibration comparison (raw cosine is a similarity, not a probability)")
display(pd.DataFrame(experiment_results["calibration_ablation"]))
print("Memory duration")
display(pd.DataFrame(experiment_results["memory_duration_ablation"]))

## 16. Calculate and inspect metrics

In [ ]:
print("Held-out candidate-pair calibration metrics")
display(pd.DataFrame([{
    key: value for key, value in experiment_results["calibration_metrics"].items()
    if key not in {"reliability"}
}]))
print("Per-sequence tracking and association metrics")
display(pd.DataFrame(experiment_results["per_sequence_metrics"]).drop(
    columns=["occlusion_gaps", "candidate_labels", "candidate_probabilities"],
    errors="ignore",
))
print("Natural object absence/reappearance analysis")
display(pd.DataFrame([
    {"method": method, **point}
    for method, points in experiment_results["occlusion_gap_metrics"].items()
    for point in points
]))

## 17. Generate publication-style plots

In [ ]:
from occlusion_association.result_io import generate_plots_from_results

preview_directory = Path("/tmp/occlusion_association_plot_preview")
preview_figures = generate_plots_from_results(experiment_results, preview_directory)
print("Preview figures:", len(preview_figures))
print("PNG and PDF preview files:", preview_figures[:6])

## 18. Export results

In [ ]:
from occlusion_association.result_io import export_experiment_results
from occlusion_association.utils import create_experiment_directory, runtime_metadata, set_random_seeds

set_random_seeds(int(config.experiment["random_seed"]))
experiment_directory = create_experiment_directory(config.experiment["output_dir"])
metadata = runtime_metadata(PROJECT_DIR)
metadata.update({
    "dataset_root": str(dataset.root),
    "dataset_split": {
        "train": train_sequence_names,
        "validation": validation_sequence_names,
    },
    "backbone": extractor.backbone_name,
    "random_seed": int(config.experiment["random_seed"]),
    "smoke_test": RUN_MODE == "smoke",
    "frame_limit": FRAME_LIMIT,
    "feature_cache_directory": str(cache.directory),
    "experiment_parameters": config.to_dict(),
})
(experiment_directory / "logs").mkdir(parents=True, exist_ok=True)
(experiment_directory / "logs" / "run.log").write_text(
    "Completed via notebooks/occlusion_aware_association.ipynb\n",
    encoding="utf-8",
)
export_experiment_results(
    experiment_results,
    experiment_directory,
    config=config.to_dict(),
    metadata=metadata,
    make_plots=True,
)
print("Exported to:", experiment_directory)

## 19. Generate experiment summary

In [ ]:
summary_columns = [
    "method",
    "identity_switches",
    "track_fragmentations",
    "association_precision",
    "association_recall",
    "association_coverage",
    "false_association_rate",
    "brier_score",
    "ece",
]
display(pd.DataFrame(experiment_results["metrics"])[summary_columns])
print("Calibrator training sequences:", experiment_results["train_candidate_summary"]["sequences"])
print("Validation sequences:", experiment_results["sequence_split"]["validation"])
print("Smoke test active:", experiment_results["smoke_test"])
print("Result directory:", experiment_directory)
print(
    "Limitations: ground-truth masks are detections; segmentation is not evaluated. "
    "Natural occlusion analysis uses only intervals with complete valid annotations."
)